# RFdiffusion → ProteinMPNN → Boltz-2  |  RBX1 RING Core Binder Design

**Target:** RBX1 RING core, chain B residues 1–52 (`rbx1_ring_core_renumbered.pdb`)  
**Pipeline:**
1. RFdiffusion — generate binder backbones conditioned on RING surface hotspots
2. ProteinMPNN — sample sequences at 3 temperatures × 3 seqs = 9 per backbone
3. Boltz-2 — rescore all binder+RBX1 complexes, filter by ipTM

All outputs saved to Google Drive.

In [ ]:
# ============================================================
# ⚙️  USER CONFIG — edit these before running
# ============================================================

# Google Drive folder (will be created if it doesn't exist)
DRIVE_DIR = '/content/drive/MyDrive/adaptyv_competition'

# Target PDB filename — upload to Drive DRIVE_DIR before running
TARGET_PDB_NAME = 'rbx1_ring_core_renumbered.pdb'

# RFdiffusion: receptor chain and residue range
TARGET_CHAIN   = 'B'
TARGET_RES     = '1-52'          # residue range in the PDB

# RFdiffusion: contigs — fix target, diffuse binder of 65-95 AA
CONTIGS        = f'{TARGET_CHAIN}{TARGET_RES}/0 65-95'

# RFdiffusion: hotspot residues on the RING-H2 E2-binding surface
# (1-based in the 52 AA fragment: H2 loop + zinc-coordinating surface)
HOTSPOT_RESIDUES = f'{TARGET_CHAIN}4,{TARGET_CHAIN}6,{TARGET_CHAIN}12,{TARGET_CHAIN}14,{TARGET_CHAIN}15,{TARGET_CHAIN}23,{TARGET_CHAIN}24,{TARGET_CHAIN}26,{TARGET_CHAIN}28,{TARGET_CHAIN}52'

# Number of RFdiffusion backbones to generate
NUM_DESIGNS    = 100

# ProteinMPNN sampling
MPNN_TEMPS     = [0.05, 0.10, 0.15]   # sequence diversity temperatures
MPNN_SEQS_PER_TEMP = 3                 # sequences per temperature per backbone

# Boltz-2 filter threshold
IPTM_THRESHOLD = 0.65

# RBX1 RING core sequence for Boltz-2 complex YAMLs
RBX1_RING_CORE = 'LWAWDIVVDNCAICRNHIMDLCIECQANQASATSEECTVAWGVCNHAFHFHC'  # res 32-83, 52 AA

print('Config loaded')
print(f'  Contigs:   {CONTIGS}')
print(f'  Hotspots:  {HOTSPOT_RESIDUES}')
print(f'  Designs:   {NUM_DESIGNS}')
print(f'  MPNN seqs: {NUM_DESIGNS} backbones × {len(MPNN_TEMPS)} temps × {MPNN_SEQS_PER_TEMP} = {NUM_DESIGNS * len(MPNN_TEMPS) * MPNN_SEQS_PER_TEMP} total')

In [ ]:
# ============================================================
# 📁  Mount Google Drive + set up output directories
# ============================================================
import os
from google.colab import drive

drive.mount('/content/drive')

RFD_OUT   = f'{DRIVE_DIR}/rfd_outputs'
MPNN_OUT  = f'{DRIVE_DIR}/mpnn_outputs'
BOLTZ_OUT = f'{DRIVE_DIR}/boltz_outputs'
YAML_DIR  = f'{DRIVE_DIR}/boltz_yamls'
FINAL_OUT = f'{DRIVE_DIR}/final'

for d in [RFD_OUT, MPNN_OUT, BOLTZ_OUT, YAML_DIR, FINAL_OUT]:
    os.makedirs(d, exist_ok=True)

TARGET_PDB = f'{DRIVE_DIR}/{TARGET_PDB_NAME}'
assert os.path.isfile(TARGET_PDB), (
    f'Target PDB not found: {TARGET_PDB}\n'
    f'Upload {TARGET_PDB_NAME} to your Drive folder {DRIVE_DIR} and re-run.'
)
print(f'✓ Target PDB found: {TARGET_PDB}')
print(f'  Outputs → {DRIVE_DIR}/')

In [ ]:
# ============================================================
# 🛠️  Install RFdiffusion + dependencies
# Matches the approach from the previous working run.
# ============================================================
import os, subprocess, glob, re, py_compile, tempfile

def run(name, cmd):
    r = subprocess.run(cmd, shell=True, cwd='/content', capture_output=True, text=True)
    print(f"{'OK' if r.returncode == 0 else 'FAIL'}: {name}")
    if r.returncode != 0:
        print(r.stderr[-300:])

if not os.path.isdir('/content/RFdiffusion'):
    run('Clone RFdiffusion', 'git clone -q https://github.com/RosettaCommons/RFdiffusion.git /content/RFdiffusion')

# Use lucidrains se3-transformer-pytorch — avoids the DGL CUDA "Operator Range" crash
run('Install se3-transformer-pytorch', 'pip -q install git+https://github.com/lucidrains/se3-transformer-pytorch')
# Install RFdiffusion — pulls in DGL from PyPI (CPU-compatible version)
run('Install RFdiffusion', 'pip -q install -e /content/RFdiffusion')
run('Install omegaconf + pyrsistent', 'pip -q install omegaconf pyrsistent')

# ── graphbolt patch (in case DGL installs a version that needs it) ─────────────
gb_inits = glob.glob('/usr/local/lib/python*/dist-packages/dgl/graphbolt/__init__.py')
if gb_inits:
    path = gb_inits[0]
    with open(path) as f:
        content = f.read()
    content = re.sub(
        r'^def load_graphbolt\b[^\n]*\n(?:(?!^(?:def |class |\S)).*\n)*',
        '', content, flags=re.MULTILINE,
    )
    content = re.sub(r'^\s*load_graphbolt\s*\([^)]*\)\s*\n', '', content, flags=re.MULTILINE)
    with open(path, 'w') as f:
        f.write(content)
    try:
        tmp = tempfile.NamedTemporaryFile(suffix='.py', delete=False, mode='w')
        tmp.write(content); tmp.close()
        py_compile.compile(tmp.name, doraise=True)
        print('✓ DGL graphbolt patched — syntax OK')
    except py_compile.PyCompileError:
        with open(path, 'w') as f:
            f.write('# graphbolt disabled\n')
        print('✓ DGL graphbolt blanked (fallback)')
# ──────────────────────────────────────────────────────────────────────────────

print('\nVerifying imports...')
import importlib
for pkg in ['dgl', 'e3nn', 'hydra', 'pyrsistent']:
    try:
        importlib.import_module(pkg)
        print(f'  {pkg} ✓')
    except Exception as e:
        print(f'  {pkg} FAILED: {e}')
print('Done.')

In [ ]:
# ============================================================
# 📥  Download RFdiffusion model weights (to /content, not Drive)
# ============================================================
import os, subprocess

WEIGHTS_DIR = '/content/RFdiffusion/models'
os.makedirs(WEIGHTS_DIR, exist_ok=True)

weights = {
    'Base_ckpt.pt':         'https://files.ipd.uw.edu/pub/RFdiffusion/6f5902ac237024bdd0c176cb93063dc6/Base_ckpt.pt',
    'Complex_base_ckpt.pt': 'https://files.ipd.uw.edu/pub/RFdiffusion/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt',
}

for fname, url in weights.items():
    dest = f'{WEIGHTS_DIR}/{fname}'
    if os.path.isfile(dest):
        print(f'  {fname} already downloaded')
        continue
    print(f'  Downloading {fname}...')
    r = subprocess.run(f'wget -q -O {dest} {url}', shell=True)
    print(f'  {fname} ✓' if r.returncode == 0 else f'  {fname} FAILED')

In [ ]:
# ============================================================
# 🧬  Run RFdiffusion
# ============================================================
import os, subprocess, shutil, glob

# Copy target PDB to local /content
LOCAL_TARGET = f'/content/{TARGET_PDB_NAME}'
shutil.copy(TARGET_PDB, LOCAL_TARGET)
print(f'Target PDB copied to {LOCAL_TARGET}')

LOCAL_RFD_OUT = '/content/rfd_outputs'
os.makedirs(LOCAL_RFD_OUT, exist_ok=True)

cmd = (
    f"python /content/RFdiffusion/scripts/run_inference.py"
    f" inference.input_pdb={LOCAL_TARGET}"
    f" inference.output_prefix={LOCAL_RFD_OUT}/binder"
    f" inference.num_designs={NUM_DESIGNS}"
    f" 'contigmap.contigs=[{CONTIGS}]'"
    f" 'ppi.hotspot_res=[{HOTSPOT_RESIDUES}]'"
    f" inference.ckpt_override_path=/content/RFdiffusion/models/Complex_base_ckpt.pt"
    f" denoiser.noise_scale_ca=1.0"
    f" denoiser.noise_scale_frame=1.0"
)

print('Running RFdiffusion...')
print(f'  Contigs:  {CONTIGS}')
print(f'  Hotspots: {HOTSPOT_RESIDUES}')
print(f'  Designs:  {NUM_DESIGNS}')

env = os.environ.copy()
env['PYTHONPATH'] = '/content/RFdiffusion:' + env.get('PYTHONPATH', '')

result = subprocess.run(cmd, shell=True, cwd='/content', env=env, text=True,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(result.stdout[-3000:] if result.stdout else '(no output)')
if result.returncode != 0:
    print(f'\n⚠ RFdiffusion exited with code {result.returncode}')

# Sync outputs to Drive
rfd_pdbs = sorted(glob.glob(f'{LOCAL_RFD_OUT}/binder_*.pdb'))
print(f'\nRFdiffusion produced {len(rfd_pdbs)} PDB files')
for pdb in rfd_pdbs:
    shutil.copy(pdb, f'{RFD_OUT}/{os.path.basename(pdb)}')
for trb in glob.glob(f'{LOCAL_RFD_OUT}/binder_*.trb'):
    shutil.copy(trb, f'{RFD_OUT}/{os.path.basename(trb)}')
print(f'✓ Saved to {RFD_OUT}')

In [ ]:
# ============================================================
# 📊  Parse RFdiffusion outputs — extract binder lengths & pLDDT
# ============================================================
import glob, json, pickle
import numpy as np

rfd_pdbs = sorted(glob.glob(f'{LOCAL_RFD_OUT}/binder_*.pdb'))

rfd_entries = []
for pdb in rfd_pdbs:
    name = os.path.basename(pdb).replace('.pdb', '')
    trb  = pdb.replace('.pdb', '.trb')

    # Count residues on chain A (binder)
    residues = set()
    with open(pdb) as f:
        for line in f:
            if line.startswith('ATOM') and line[21] == 'A':
                residues.add(int(line[22:26]))
    length = len(residues)

    # Parse pLDDT from trb
    mean_plddt = None
    if os.path.isfile(trb):
        try:
            with open(trb, 'rb') as f:
                trb_data = pickle.load(f)
            plddt = trb_data.get('plddt', [])
            if len(plddt):
                mean_plddt = float(np.mean(plddt))
        except Exception:
            pass

    rfd_entries.append({'name': name, 'pdb': pdb, 'length': length, 'plddt': mean_plddt})

# Filter by length (65-95 AA for binder)
valid = [e for e in rfd_entries if 65 <= e['length'] <= 95]
print(f'Total backbones:          {len(rfd_entries)}')
print(f'Pass length filter 65-95: {len(valid)}')
if valid and valid[0]['plddt'] is not None:
    plddts = [e['plddt'] for e in valid if e['plddt'] is not None]
    print(f'Mean pLDDT (binder):      {np.mean(plddts):.3f}')

In [ ]:
# ============================================================
# 🛠️  Install ProteinMPNN
# ============================================================
import os, subprocess

if not os.path.isdir('/content/ProteinMPNN'):
    r = subprocess.run('git clone -q https://github.com/dauparas/ProteinMPNN.git /content/ProteinMPNN',
                       shell=True, capture_output=True, text=True)
    print('ProteinMPNN cloned' if r.returncode == 0 else r.stderr)
else:
    print('ProteinMPNN already present')

import sys
sys.path.append('/content/ProteinMPNN')

import torch
from protein_mpnn_utils import (
    _S_to_seq, tied_featurize, parse_PDB,
    StructureDatasetPDB, ProteinMPNN
)
import copy, numpy as np

# Load ProteinMPNN model
mpnn_device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
ckpt_path   = '/content/ProteinMPNN/vanilla_model_weights/v_48_020.pt'
ckpt        = torch.load(ckpt_path, map_location=mpnn_device)
mpnn_model  = ProteinMPNN(
    num_letters=21, node_features=128, edge_features=128, hidden_dim=128,
    num_encoder_layers=3, num_decoder_layers=3, augment_eps=0.0,
    k_neighbors=ckpt['num_edges']
)
mpnn_model.to(mpnn_device)
mpnn_model.load_state_dict(ckpt['model_state_dict'])
mpnn_model.eval()
print(f'ProteinMPNN loaded on {mpnn_device}')

In [ ]:
# ============================================================
# 🔬  ProteinMPNN — 9 sequences per backbone (3 temps × 3 seqs)
# Design chain A (binder), fix chain B (RBX1 target)
# ============================================================
import os, copy, torch, numpy as np
from protein_mpnn_utils import _S_to_seq, tied_featurize, parse_PDB, StructureDatasetPDB

alphabet     = 'ACDEFGHIKLMNPQRSTVWYX'
omit_AAs_np  = np.array([aa in 'X' for aa in alphabet], dtype=np.float32)
bias_AAs_np  = np.zeros(len(alphabet))

mpnn_seqs = []

for entry in valid:
    pdb_path = entry['pdb']
    try:
        pdb_dict_list = parse_PDB(pdb_path, input_chain_list=['A', 'B'])
    except Exception as e:
        print(f'  SKIP {entry["name"]}: parse failed — {e}')
        continue
    if not pdb_dict_list:
        print(f'  SKIP {entry["name"]}: empty parse')
        continue

    # Chain A = binder (design), Chain B = RBX1 (fixed)
    chain_id_dict = {pdb_dict_list[0]['name']: (['A'], ['B'])}
    binder_len    = len(pdb_dict_list[0].get('seq_chain_A', ''))

    for temp in MPNN_TEMPS:
        with torch.no_grad():
            for protein in StructureDatasetPDB(pdb_dict_list, truncate=None, max_length=20000):
                batch = [copy.deepcopy(protein) for _ in range(MPNN_SEQS_PER_TEMP)]
                try:
                    (
                        X, S, mask, lengths, chain_M, chain_encoding_all,
                        chain_list_list, visible_list_list, masked_list_list,
                        masked_chain_length_list_list, chain_M_pos,
                        omit_AA_mask, residue_idx, dihedral_mask,
                        tied_pos_list_of_lists_list, pssm_coef, pssm_bias,
                        pssm_log_odds_all, bias_by_res_all, tied_beta
                    ) = tied_featurize(batch, mpnn_device, chain_id_dict,
                                       None, None, None, None, None)
                except Exception as e:
                    print(f'  SKIP {entry["name"]} T={temp}: featurize failed — {e}')
                    continue

                sample_dict = mpnn_model.sample(
                    X,
                    torch.randn(chain_M.shape, device=mpnn_device),
                    S, chain_M, chain_encoding_all, residue_idx,
                    mask=mask, temperature=temp,
                    omit_AAs_np=omit_AAs_np, bias_AAs_np=bias_AAs_np,
                    chain_M_pos=chain_M_pos, omit_AA_mask=omit_AA_mask,
                    pssm_coef=pssm_coef, pssm_bias=pssm_bias,
                    pssm_multi=0.0, pssm_log_odds_flag=False,
                    pssm_log_odds_mask=(pssm_log_odds_all > 0.0).float(),
                    pssm_bias_flag=False, bias_by_res=bias_by_res_all
                )

                for b_ix in range(MPNN_SEQS_PER_TEMP):
                    full_seq = _S_to_seq(sample_dict['S'][b_ix], chain_M[b_ix])
                    seq      = full_seq[:binder_len]
                    # Skip degenerate sequences
                    if not seq or len(seq) < 30:
                        continue
                    # Skip poly-Ala artefacts (>40% Ala)
                    if seq.count('A') / len(seq) > 0.4:
                        continue
                    mpnn_seqs.append({
                        'name':     f"RFD_{entry['name'].split('_')[-1]}_t{int(temp*100):03d}_s{b_ix+1}",
                        'seq':      seq,
                        'length':   len(seq),
                        'backbone': entry['name'],
                        'temp':     temp,
                        'rfd_plddt': entry.get('plddt'),
                    })

# Deduplicate by sequence
seen = set()
unique_seqs = []
for d in mpnn_seqs:
    if d['seq'] not in seen:
        seen.add(d['seq'])
        unique_seqs.append(d)

print(f'ProteinMPNN generated: {len(mpnn_seqs)} sequences')
print(f'After deduplication:   {len(unique_seqs)} unique')

# Save FASTA to Drive
mpnn_fasta = f'{MPNN_OUT}/mpnn_candidates.fasta'
with open(mpnn_fasta, 'w') as f:
    for d in unique_seqs:
        f.write(f">{d['name']} len={d['length']} rfd_plddt={d['rfd_plddt']:.3f}\n{d['seq']}\n")
print(f'Saved: {mpnn_fasta}')

In [ ]:
# ============================================================
# 📄  Write Boltz-2 input YAMLs (binder + RBX1 complex)
# ============================================================
import os

yaml_tmpl = (
    'version: 1\n'
    'sequences:\n'
    '  - protein:\n'
    '      id: A\n'
    '      sequence: {binder}\n'
    '  - protein:\n'
    '      id: B\n'
    '      sequence: {rbx1}\n'
)

for d in unique_seqs:
    yaml_path = f"{YAML_DIR}/{d['name']}.yaml"
    with open(yaml_path, 'w') as f:
        f.write(yaml_tmpl.format(binder=d['seq'], rbx1=RBX1_RING_CORE))

print(f'Written {len(unique_seqs)} YAML files to {YAML_DIR}')

In [ ]:
# ============================================================
# 🛠️  Install Boltz-2
# ============================================================
import subprocess, importlib

try:
    import boltz
    print(f'Boltz already installed: {boltz.__version__}')
except ImportError:
    print('Installing boltz...')
    r = subprocess.run('pip install -q boltz', shell=True,
                       capture_output=True, text=True)
    print('Installed' if r.returncode == 0 else r.stderr[-300:])

# Pre-download Boltz-2 weights to /content (not Drive)
import subprocess
r = subprocess.run('boltz download', shell=True, capture_output=True, text=True)
print(r.stdout[-200:] if r.stdout else 'Weights ready')

In [ ]:
# ============================================================
# 🔬  Boltz-2 complex rescoring
# ============================================================
import os, glob, subprocess

yaml_files = sorted(glob.glob(f'{YAML_DIR}/*.yaml'))
print(f'Rescoring {len(yaml_files)} complexes with Boltz-2...')

for i, yf in enumerate(yaml_files):
    name     = os.path.basename(yf).replace('.yaml', '')
    out_dir  = f'{BOLTZ_OUT}/{name}'
    # Skip if already scored
    done = glob.glob(f'{out_dir}/**/confidence_*.json', recursive=True)
    if done:
        continue
    os.makedirs(out_dir, exist_ok=True)
    cmd = (
        f'boltz predict {yf} --out_dir {out_dir} '
        f'--accelerator gpu '
        f'--recycling_steps 3 '
        f'--sampling_steps 200 '
        f'--diffusion_samples 1 '
        f'--no_kernels '
        f'--num_workers 2 2>/dev/null'
    )
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if (i + 1) % 20 == 0:
        print(f'  {i+1}/{len(yaml_files)} done')

print('Boltz-2 rescoring complete')

In [ ]:
# ============================================================
# 🏆  Collect scores, rank, filter
# ============================================================
import json, glob

scored = []
for d in unique_seqs:
    conf_files = glob.glob(f"{BOLTZ_OUT}/{d['name']}/**/confidence_*.json", recursive=True)
    if not conf_files:
        continue
    with open(conf_files[0]) as f:
        conf = json.load(f)
    scored.append({
        **d,
        'iptm':   conf.get('iptm', 0.0),
        'iplddt': conf.get('interface_plddt', conf.get('complex_plddt', 0.0)),
        'ptm':    conf.get('ptm', 0.0),
    })

scored.sort(key=lambda r: -r['iptm'])
passing    = [r for r in scored if r['iptm'] >= IPTM_THRESHOLD]
borderline = [r for r in scored if 0.60 <= r['iptm'] < IPTM_THRESHOLD]

print(f'Scored:            {len(scored)}')
print(f'PASS  (≥{IPTM_THRESHOLD}):   {len(passing)}')
print(f'BORDER (≥0.60):    {len(borderline)}')
print(f'FAIL  (<0.60):     {len(scored)-len(passing)-len(borderline)}')

print(f'\nTop 20 by Boltz-2 ipTM:')
print(f'{"name":<35} {"ipTM":>6} {"ipLDDT":>7} {"pTM":>5} {"len":>5}')
print('-' * 60)
for r in scored[:20]:
    flag = ' ✓' if r['iptm'] >= IPTM_THRESHOLD else (' ~' if r['iptm'] >= 0.60 else '')
    print(f"{r['name']:<35} {r['iptm']:>6.3f} {r['iplddt']:>7.3f} {r['ptm']:>5.3f} {r['length']:>5}{flag}")

In [ ]:
# ============================================================
# 💾  Export FASTA + CSV to Drive
# ============================================================
import csv as _csv

keep = passing + borderline

fasta_path = f'{FINAL_OUT}/rfd_mpnn_boltz_passing.fasta'
csv_path   = f'{FINAL_OUT}/rfd_mpnn_boltz_passing.csv'

with open(fasta_path, 'w') as f:
    for r in keep:
        f.write(f">{r['name']} iptm={r['iptm']:.3f} iplddt={r['iplddt']:.3f} len={r['length']}\n{r['seq']}\n")

fields = ['name', 'length', 'iptm', 'iplddt', 'ptm', 'backbone', 'temp', 'rfd_plddt', 'seq']
with open(csv_path, 'w', newline='') as f:
    w = _csv.DictWriter(f, fieldnames=fields, extrasaction='ignore')
    w.writeheader()
    w.writerows(keep)

print(f'Exported {len(keep)} sequences (PASS + BORDERLINE)')
print(f'  FASTA: {fasta_path}')
print(f'  CSV:   {csv_path}')

if keep:
    print(f'\nipTM range:    {min(r["iptm"] for r in keep):.3f} – {max(r["iptm"] for r in keep):.3f}')
    print(f'Length range:  {min(r["length"] for r in keep)} – {max(r["length"] for r in keep)} AA')
    unique_backbones = len(set(r['backbone'] for r in keep))
    print(f'Unique backbones represented: {unique_backbones}')

In [ ]:
# ============================================================
# 📦  Download results to local machine
# ============================================================
from google.colab import files
files.download(fasta_path)
files.download(csv_path)